# Análise estatística pareada dos modelos M0, M1 e M2

Este notebook reproduz a comparação estatística dos **erros absolutos diários** das três configurações avaliadas no TCC.

Procedimento:

1. leitura da tabela diária gerada pelo pipeline principal;
2. teste não paramétrico de Friedman;
3. comparações pareadas pelo teste de Wilcoxon;
4. correção de Holm para três comparações;
5. cálculo opcional de Kendall's W como medida global de efeito;
6. exportação de uma tabela-resumo.

A unidade pareada é o **dia**. Portanto, não são testados diretamente os valores globais de MAE ou RMSE.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import friedmanchisquare, wilcoxon


## Configuração

Por padrão, o notebook procura o arquivo de resultados produzido pelo notebook de modelagem em um caminho relativo ao repositório.

Caso a estrutura local seja diferente, altere apenas `ARQUIVO_RESULTADOS`.


In [ ]:
ARQUIVO_RESULTADOS = Path("../results/Indicadores_Modelos_5min_GOES_DEFINITIVO.xlsx")

# Caso o arquivo esteja em outro local, ajuste a linha acima.
print("Arquivo esperado:", ARQUIVO_RESULTADOS.resolve())


## Leitura da tabela de erros diários

O pipeline definitivo exporta uma planilha com os resultados diários. Esta célula procura automaticamente uma aba cujo nome contenha "erro" ou "diar".


In [ ]:
if not ARQUIVO_RESULTADOS.exists():
    raise FileNotFoundError(
        f"Arquivo não encontrado: {ARQUIVO_RESULTADOS}\n"
        "Ajuste ARQUIVO_RESULTADOS para o local da planilha gerada pelo pipeline principal."
    )

xls = pd.ExcelFile(ARQUIVO_RESULTADOS)
abas = xls.sheet_names
print("Abas disponíveis:", abas)

candidatas = [s for s in abas if ("erro" in s.lower() or "diar" in s.lower())]
if not candidatas:
    raise ValueError(
        "Não foi encontrada automaticamente uma aba de erros diários. "
        "Defina manualmente o nome da aba."
    )

ABA = candidatas[0]
df = pd.read_excel(ARQUIVO_RESULTADOS, sheet_name=ABA)

print("Aba selecionada:", ABA)
print("Número de linhas:", len(df))
display(df.head())


## Identificação das colunas de erro absoluto

O notebook procura colunas contendo simultaneamente os termos `abs`, `erro` e o identificador do modelo (`M0`, `M1` ou `M2`).

Se os nomes forem alterados no futuro, a função abaixo pode ser ajustada sem modificar o restante da análise.


In [ ]:
def localizar_coluna_abs_erro(colunas, modelo):
    modelo = modelo.lower()
    candidatos = []
    for c in colunas:
        cl = str(c).lower()
        if "abs" in cl and "erro" in cl and modelo in cl:
            candidatos.append(c)
    if len(candidatos) != 1:
        raise ValueError(
            f"Esperava exatamente uma coluna de erro absoluto para {modelo.upper()}, "
            f"mas encontrei: {candidatos}"
        )
    return candidatos[0]

COL_M0 = localizar_coluna_abs_erro(df.columns, "m0")
COL_M1 = localizar_coluna_abs_erro(df.columns, "m1")
COL_M2 = localizar_coluna_abs_erro(df.columns, "m2")

print("M0:", COL_M0)
print("M1:", COL_M1)
print("M2:", COL_M2)


In [ ]:
dados = df[[COL_M0, COL_M1, COL_M2]].apply(pd.to_numeric, errors="coerce").dropna()

m0 = dados[COL_M0].to_numpy(float)
m1 = dados[COL_M1].to_numpy(float)
m2 = dados[COL_M2].to_numpy(float)

n = len(dados)
print("Dias válidos pareados:", n)

if n < 3:
    raise ValueError("Número insuficiente de observações pareadas.")


## Teste de Friedman

Hipótese nula: não há diferença sistemática entre os três modelos quanto aos erros absolutos diários.

Adota-se nível de significância de 5%.


In [ ]:
ALPHA = 0.05

friedman_stat, friedman_p = friedmanchisquare(m0, m1, m2)

# Kendall's W para k = 3 tratamentos
k = 3
kendall_w = friedman_stat / (n * (k - 1))

print(f"Friedman: chi2 = {friedman_stat:.6f}")
print(f"p = {friedman_p:.6g}")
print(f"Kendall's W = {kendall_w:.6f}")
print("Resultado:", "significativo" if friedman_p < ALPHA else "não significativo")


## Comparações pareadas por Wilcoxon

São avaliadas as comparações M0×M1, M0×M2 e M1×M2.

Como podem existir diferenças exatamente iguais a zero, utiliza-se `zero_method="wilcox"`, que descarta pares com diferença zero conforme a implementação padrão do SciPy.


In [ ]:
pares = {
    "M0 × M1": (m0, m1),
    "M0 × M2": (m0, m2),
    "M1 × M2": (m1, m2),
}

resultados = []

for nome, (a, b) in pares.items():
    stat, p = wilcoxon(
        a,
        b,
        zero_method="wilcox",
        alternative="two-sided",
        method="auto",
    )
    resultados.append({
        "comparacao": nome,
        "estatistica_W": float(stat),
        "p_bruto": float(p),
    })

res = pd.DataFrame(resultados)
display(res)


## Correção de Holm

A correção de Holm controla o erro familiar das três comparações pós-hoc sem assumir independência entre os testes.


In [ ]:
def holm_adjust(pvalues):
    pvalues = np.asarray(pvalues, dtype=float)
    m = len(pvalues)
    ordem = np.argsort(pvalues)
    ajustados_ordenados = np.empty(m, dtype=float)

    max_anterior = 0.0
    for rank, idx in enumerate(ordem):
        fator = m - rank
        valor = min(1.0, fator * pvalues[idx])
        valor = max(valor, max_anterior)
        ajustados_ordenados[rank] = valor
        max_anterior = valor

    ajustados = np.empty(m, dtype=float)
    for rank, idx in enumerate(ordem):
        ajustados[idx] = ajustados_ordenados[rank]

    return ajustados

res["p_ajustado_holm"] = holm_adjust(res["p_bruto"].to_numpy())
res["resultado_5pct"] = np.where(
    res["p_ajustado_holm"] < ALPHA,
    "Significativo",
    "Não significativo"
)

display(res)


## Tabela final

A tabela abaixo reúne o teste global e os testes pós-hoc.


In [ ]:
linha_friedman = pd.DataFrame([{
    "comparacao": "M0, M1 e M2",
    "teste": "Friedman",
    "estatistica": friedman_stat,
    "p_bruto": friedman_p,
    "p_ajustado_holm": np.nan,
    "resultado_5pct": "Significativo" if friedman_p < ALPHA else "Não significativo",
}])

linhas_wilcoxon = res.rename(columns={"estatistica_W": "estatistica"}).copy()
linhas_wilcoxon["teste"] = "Wilcoxon"
linhas_wilcoxon = linhas_wilcoxon[
    ["comparacao", "teste", "estatistica", "p_bruto", "p_ajustado_holm", "resultado_5pct"]
]

tabela_final = pd.concat([linha_friedman, linhas_wilcoxon], ignore_index=True)
display(tabela_final)


## Exportação

Os resultados são salvos em CSV para facilitar a inclusão em tabelas, apêndices ou auditorias futuras.


In [ ]:
SAIDA = Path("../results/testes_estatisticos_M0_M1_M2.csv")
SAIDA.parent.mkdir(parents=True, exist_ok=True)

tabela_final.to_csv(SAIDA, index=False, encoding="utf-8-sig")
print("Arquivo salvo em:", SAIDA.resolve())


## Observação metodológica

Os testes acima comparam os erros absolutos diários observados nos mesmos dias.

A interpretação inferencial deve considerar que os dados são temporais e que as previsões OOF derivam de folds com conjuntos de treinamento parcialmente sobrepostos. Assim, os testes devem ser entendidos como uma comparação pareada complementar ao desempenho preditivo, e não como uma prova isolada de superioridade universal de um modelo.
